In [5]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt
from collections import defaultdict

MODULE_DIR = Path.cwd()
if not (MODULE_DIR / "fluent_deeponet.py").exists():
    MODULE_DIR = MODULE_DIR / "DeepONet"
ROOT_DIR = MODULE_DIR.parent

if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from eval_fluent_deeponet import (
    make_plot_dict_from_iterative_result,
    select_sample_indices_by_case_id,
    infer_edge_layout,
    PhysicsInterfaceConfig,
    physics_unknown_interface_inference,
)
from fluent_deeponet import DeepONet, FeatureNormalizer
from deeponet_fluent_dataset import build_fluent_deeponet_dataset

from plot import plot_prediction_imshow_from_points

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
device = torch.device("cuda:2")
ckpt_path = Path("DeepONet/results") / "080426_1" / "checkpoint.pt"
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

model = DeepONet(**ckpt["model_config"]).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

y_normalizer = FeatureNormalizer.from_state_dict(ckpt["y_normalizer"]).to(device)
local_aspect_mean = float(ckpt["local_aspect_mean"])
local_aspect_std = float(ckpt["local_aspect_std"])


In [27]:
FIELD_MAP = {"pressure": "SV_P", "u": "SV_U", "v": "SV_V"}
ROOT_DIR = Path("/home/hantianl/Documents/PIDIF/")
DATASET_NAME = "channel_water"

def case_paths(ch):
    return {
        "design": ROOT_DIR / f"2d_geometry_specs/{DATASET_NAME}/{ch}.json",
        "mesh": ROOT_DIR / f"runs_2d/{DATASET_NAME}/{ch}/{ch}.msh.h5",
        "dat": ROOT_DIR / f"runs_2d/{DATASET_NAME}/{ch}/case2d.dat.h5",
    }

available_cases = [
    ch.name for ch in (ROOT_DIR / "runs_2d" / DATASET_NAME).iterdir()
    if all(case_paths(ch.name)[k].exists() for k in ("design", "mesh", "dat"))
]
case_files = {ch: case_paths(ch) for ch in available_cases}

bc_kwargs = {
    "inlet_v": 0.0,
    "outlet_p": 0.0,
    "wall_u": 0.0,
    "wall_v": 0.0,
}

# TEST_CH = ['channel_18', 'channel_22', 'channel_28', 'channel_31', 'channel_33', 'channel_45', 'channel_51', 'channel_71', 
#            'channel_81', 'channel_89', 'channel_90', 'channel_97', 'channel_111', 'channel_117', 'channel_118', 'channel_133', 
#            'channel_151', 'channel_172', 'channel_193', 'channel_199']
# TEST_CH = ['channel_00', 'channel_01', 'channel_02', 'channel_03', 'channel_04', 'channel_05', 'channel_06', 'channel_07', 
#            'channel_08', 'channel_09']
TEST_CH = [f"channel_{i:03d}" for i in range(20)] + [f"channel_{i:03d}" for i in range(40, 50)] + [f"channel_{i:03d}" for i in range(210, 220)] + [f"channel_{i:03d}" for i in range(320, 330)]
rng = np.random.default_rng(0)

test_data = build_fluent_deeponet_dataset(
    case_files=case_files,
    case_ids=TEST_CH,
    n_subdomains="adaptive",
    n_interface_points=256,
    n_boundary_points=256,
    interface_placement="fixed",
    interface_jitter=0,
    insert_sharp_control_point_interfaces=False,
    field_map=FIELD_MAP,
    bc_kwargs=bc_kwargs,
    keep_raw_case_data=False,
    include_reynolds=False,
    include_sdf=False,
    rng=rng,
)

sample_test_ids = select_sample_indices_by_case_id(test_data, case_id=TEST_CH[0])

Setting n_realizations to 1 for fixed interface placement with no jitter
Processing case channel_000 realization=0 (n_subdomains=5)
Processing case channel_001 realization=0 (n_subdomains=5)
Processing case channel_002 realization=0 (n_subdomains=5)
Processing case channel_003 realization=0 (n_subdomains=5)
Processing case channel_004 realization=0 (n_subdomains=5)
Processing case channel_005 realization=0 (n_subdomains=5)
Processing case channel_006 realization=0 (n_subdomains=5)
Processing case channel_007 realization=0 (n_subdomains=5)
Processing case channel_008 realization=0 (n_subdomains=5)
Processing case channel_009 realization=0 (n_subdomains=5)
Processing case channel_010 realization=0 (n_subdomains=20)
Processing case channel_011 realization=0 (n_subdomains=20)
Processing case channel_012 realization=0 (n_subdomains=20)
Processing case channel_013 realization=0 (n_subdomains=20)
Processing case channel_014 realization=0 (n_subdomains=20)
Processing case channel_015 realizati

# Physics inference

In [33]:
results = defaultdict(dict)
plot_dicts = defaultdict(dict)
for ch in TEST_CH:
    print(f"Physics inference for {ch}")
    sample_test_ids = select_sample_indices_by_case_id(test_data, case_id=ch)

    samples = [test_data["samples"][i] for i in sample_test_ids]
    metadata = [test_data["metadata"][i] for i in sample_test_ids]

    # for seed in rng.choice(range(1000), size=5, replace=False):
    for seed in [0]:
        print(f"Seed: {seed}")
        phys_config = PhysicsInterfaceConfig(
            max_iter=0,
            lr=1e-2,
            optimizer="lbfgs",
            tol=1e-6,
            random_seed=seed,
            init_mode="fixed",
            init_value=[0.0, 0.0, 0.0],  # [pressure, u, v] for 3-output model
            init_noise_std=0.02,
            optimize_fields=["pressure", "u", "v"],
            viscosity=1.003E-3,
            length_unit_scale=1e-3,  # metadata unit is mm; converts to m
            alpha_traction=0.1,
            alpha_flux=0.3,
            alpha_dirichlet=10,
            alpha_smooth=1e-4,
            alpha_value_l2=0,
            alpha_p=10,
            alpha_u=3,
            alpha_v=0.1,
            optimize_pressure_offsets=False,
            query_batch_size=32768,
            verbose=True,
            verbose_every=25,
        )

        result_phys = physics_unknown_interface_inference(
            model=model,
            samples=samples,
            branch_channel_names=test_data["branch_channel_names"],
            output_channel_names=test_data["output_channel_names"],
            device=device,
            y_normalizer=y_normalizer,
            metadata=metadata,
            local_aspect_mean=local_aspect_mean,
            local_aspect_std=local_aspect_std,
            config=phys_config,
        )
        results[ch][seed] = result_phys

        print("Iterations:", result_phys["n_iter"])
        # print("Final physics losses:", result_phys["physics_loss_history"][-1])

        plot_dict = make_plot_dict_from_iterative_result(
            result_phys,
            test_data,
            sample_test_ids,
            output_channel_names=test_data["output_channel_names"],
        )
        plot_dicts[ch][seed] = plot_dict

        pred = plot_dict["pred"]
        truth = plot_dict["truth"]
        err = np.abs(pred - truth)

        rel_l2 = np.linalg.norm(err, axis=0) / np.linalg.norm(truth, axis=0)
        print(f"Relative L2 error: {rel_l2}, mean: {np.mean(rel_l2)}")
        print(f"Max absolute error: {np.abs(plot_dict['pred'] - plot_dict['truth']).max(axis=0)}")

Physics inference for channel_000
Seed: 635
iter=0001 | loss=7.625685e+01 | traction=2.831478e-01 | flux=2.149524e-01 | boundary=7.616395e+00 | smooth=1.008062e+00
iter=0025 | loss=1.015104e-01 | traction=3.987127e-04 | flux=1.632277e-04 | boundary=1.014216e-02 | smooth=4.121816e-06
iter=0027 | loss=1.015104e-01 | traction=3.987127e-04 | flux=1.632277e-04 | boundary=1.014216e-02 | smooth=4.121816e-06
iter=0027 | stopping: loss unchanged for 10 iterations
Iterations: 27
Final physics losses: {'loss': 0.1015104278922081, 'traction': 0.0003987126692663878, 'flux': 0.00016322769806720316, 'dirichlet': 0.010142158716917038, 'smooth': 4.121816346014384e-06, 'value_l2': 0.4495697617530823, 'max_abs_traction_res': 17.81427001953125, 'max_abs_flux_res': 6.248883437365294e-07, 'iteration': 27.0, 'traction_scale': 412.49688720703125, 'flux_scale': 4.458394414948684e-05}
Relative L2 error: [0.14183559 0.01060762 0.16333036], mean: 0.10525786131620407
Max absolute error: [5.4366748e+02 1.0311484e-0

In [ ]:
errs = []
for ch, seeds in plot_dicts.items():
    err_geo = []
    p_area = []
    for seed, pd in seeds.items():
        pred = pd["pred"]
        truth = pd["truth"]
        area = (pd["x"] > 0.1) & (pd["x"] < 0.12) & (pd["y"] > 0.04) & (pd["y"] < 0.06)
        p = pd["pred"][area, 0].mean()
        p_area.append(p)
        err = np.abs(pred - truth)
        rel_l2 = np.linalg.norm(err, axis=0) / np.linalg.norm(truth, axis=0)
        err_geo.append(rel_l2)

    p_area = np.array(p_area)
    err_geo = np.array(err_geo)
    p_mean = p_area.mean()
    p_std = p_area.std()
    # inliers = np.abs(p_area - p_mean) < p_std
    # err_geo = err_geo[inliers]
    errs.append(err_geo.mean(axis=0))

errs = np.array(errs)

array([0.04479571, 0.0110769 , 0.20310996], dtype=float32)

In [ ]:
print(errs.mean(axis=0))
print(errs.std(axis=0))
print(errs.mean())

In [ ]:
output_dir = Path("results") / "061626_2" / "predictions" / "test_pi"
output_channel_names = test_data["output_channel_names"]
for ch in TEST_CH[5:6]:
    print(f"Plotting {ch}")
    seed = 645
    pd = plot_dicts[ch][seed]
    metadatas = [test_data["metadata"][i] for i in range(len(test_data["metadata"])) if test_data["metadata"][i]["case_id"] == ch]
    for field_name in output_channel_names:
        field_idx = output_channel_names.index(field_name)
        plot_prediction_imshow_from_points(
            x=pd["x"],
            y=pd["y"],
            pred=pd["pred"],
            truth=pd["truth"],
            field_name=field_name,
            field_idx=field_idx,
            n_y_plot=200,
            metadata=metadatas, 
            mesh_h5=case_files[ch]["mesh"],
            output_dir=output_dir / f"{ch}",
        )

In [ ]:
output_channel_names = test_data["output_channel_names"]
int_id = 1
for val_id in range(3):
    br = test_data["samples"][sample_test_ids[int_id]]["branch"]
    real_t = br[(br[:, 0] == 0) & (br[:, 3] == 1)][:, val_id + 4]
    br_pred = result_phys["branch_final"][int_id]
    pred_t = br_pred[(br_pred[:, 0] == 0) & (br_pred[:, 3] == 1)][:, val_id + 4]
    pred_left = y_normalizer.decode(result_phys["pred_left_interface"]).cpu().numpy()
    pred_right = y_normalizer.decode(result_phys["pred_right_interface"]).cpu().numpy()
    pred_left = pred_left[int_id, :, val_id]
    pred_right = pred_right[int_id, :, val_id]


    plt.figure(figsize=(8, 6))
    plt.plot(real_t, zorder=10, lw=2, label="Real value", alpha=0.7)
    plt.plot(pred_t, label="Inferred interface")
    plt.plot(pred_left, label="Predicted boundary value left")
    plt.plot(pred_right, label="Predicted boundary value right")
        
    plt.xlabel("point index (0 is bottom, 256 is top)", fontsize=16)
    plt.ylabel(f"{output_channel_names[val_id]}", fontsize=16)
    plt.title(f"Physics inference (interface {int_id})")
    plt.xticks(fontsize=13)
    plt.yticks(fontsize=13)
    leg = plt.legend(loc=1, fontsize=13)
    leg.set_zorder(20)
    plt.show()
